In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

In [15]:
CSV_PATH=r"D:\Downloads\DS-ML COURSE\Jupiter(DS-ML)\raw_loan_dataset.csv"

In [16]:
# 1. Load & Inspect
df=pd.read_csv(CSV_PATH)
print(df.head(5))
print(df.info())
print(df.isnull().sum())
print("\nfound that the columns [Income, LoanAmount] string need to make them number and the other columns have words need to apply label encoding")

    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasColla

In [17]:
# 2. Clean Currency Formatting
df["Income"]=df["Income"].replace(r"[\$,]", "", regex=True).astype(float)
df["LoanAmount"]=df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)
print("After cleaned:")
print(df.info())

After cleaned:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     float64
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     float64
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Approved          103 non-null    object 
dtypes: float64(4), object(3)
memory usage: 5.8+ KB
None


In [18]:
# 3. Fix Category Errors before Imputation
yes_no_dic={"YES":"Yes", "yes":"Yes", "Y":"Yes", "y":"Yes", "yEs":"Yes", "yse":"Yes","approved":"Yes","Approved":"Yes", "1":"Yes",
           "no":"No", "NO":"No", "n":"No", "N":"No", "rejected":"No", "Rejected":"No", "0":"No"}
df["Approved"]=df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_dic).replace({"nan":np.nan})
df["PreviousDefaults"]=df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_dic).replace({"nan":np.nan})
df["HasCollateral"]=df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_dic).replace({"nan":np.nan})
print("\nAfter the fixed categories before imputation:")
print(df["Approved"].value_counts())
print("\n",df["PreviousDefaults"].value_counts())
print("\n",df["HasCollateral"].value_counts())


After the fixed categories before imputation:
Approved
Yes    68
No     35
Name: count, dtype: int64

 PreviousDefaults
No     86
Yes    15
Name: count, dtype: int64

 HasCollateral
No     51
Yes    50
Name: count, dtype: int64


In [19]:
# 4. Impute Missing Values
df["Income"]=df["Income"].fillna(df["Income"].median())
df["CreditScore"]=df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"]=df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"]=df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"]=df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"]=df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])
print("\nAfter filled missing values:\n", df.isnull().sum())


After filled missing values:
 Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64


In [20]:
# 5. Remove Duplicates
bofore_drop= df.shape
df=df.drop_duplicates()
after_drop=df.shape
print(f"Before: {bofore_drop}\nAfter: {after_drop}")

Before: (103, 7)
After: (100, 7)


In [21]:
# 6. Outliers (IQR capping)
def iqr_capping(columns, k=1.5):
    q1,q3=columns.quantile([0.25, 0.75])
    iqr= q3-q1
    low=q1-(k*iqr)
    upp=q3+(k*iqr)
    return low,upp

print("\nthe skewness of column(Income):",df["Income"].skew(),"\nneed to solve it by IQR for the outlier\n")
print("\nthe skewness of the column(CreditScore):",df["CreditScore"].skew(),"\nneed to solve it by IQR for the outlier\n")
print("\nthe skewness of column(EmploymentYears):",df["EmploymentYears"].skew(),"\nneed to solve it by IQR for the outlier\n")
print("\nthe skewness of the column(LoanAmount):",df["LoanAmount"].skew(),"\nneed to solve it by IQR for the outlier\n")

low_income, high_income=iqr_capping(df["Income"])
low_score, high_score=iqr_capping(df["CreditScore"])
lowest_year, highest_year=iqr_capping(df["EmploymentYears"])
lowest_amount,highest_amount=iqr_capping(df["LoanAmount"])
print("lowest income: ",low_income,"\nhighest income:",high_income)
print("lowest creditScore: ",low_score,"\nhighest creditScore:",high_score)
print("lowest EmploymentYears: ",lowest_year,"\nhighest EmploymentYears:",highest_year)
print("lowest LoanAmount: ",lowest_amount,"\nhighest LoanAmount:",highest_amount)

df["Income"]=df["Income"].clip(lower=low_income, upper=high_income)
df["CreditScore"]=df["CreditScore"].clip(lower=low_score, upper=high_score)
df["EmploymentYears"]=df["EmploymentYears"].clip(lower=lowest_year, upper=highest_year)
df["LoanAmount"]=df["LoanAmount"].clip(lower=lowest_amount, upper=highest_amount)
print("\nthe column(Income) after the IQR:",df["Income"].describe())
print("\nthe column(CreditScore) after the IQR:",df["CreditScore"].describe())
print("\nthe column(EmploymentYears) after the IQR:",df["EmploymentYears"].describe())
print("\nthe column(LoanAmount) after the IQR:",df["LoanAmount"].describe())


the skewness of column(Income): 1.5126310641278278 
need to solve it by IQR for the outlier


the skewness of the column(CreditScore): 0.21269591929183496 
need to solve it by IQR for the outlier


the skewness of column(EmploymentYears): -0.046570107877292634 
need to solve it by IQR for the outlier


the skewness of the column(LoanAmount): 4.088936444658662 
need to solve it by IQR for the outlier

lowest income:  -23827.875 
highest income: 167619.125
lowest creditScore:  344.25 
highest creditScore: 962.25
lowest EmploymentYears:  -11.275 
highest EmploymentYears: 35.525000000000006
lowest LoanAmount:  -16687.5 
highest LoanAmount: 66212.5

the column(Income) after the IQR: count       100.00000
mean      72220.22625
std       29179.39312
min       25851.00000
25%       47964.75000
50%       69460.50000
75%       95826.50000
max      167619.12500
Name: Income, dtype: float64

the column(CreditScore) after the IQR: count    100.000000
mean     651.100000
std       96.218239
min    

In [22]:
# 7. Label Encoding
df["Approved"]=df["Approved"].map({"Yes":1, "No":0})
df["PreviousDefaults"]=df["PreviousDefaults"].map({"Yes":1, "No":0})
df["HasCollateral"]=df["HasCollateral"].map({"Yes":1, "No":0})
print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))

Approved
1    66
0    34
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


In [23]:
# 8. Class Balance Check
class_Balance_check=df["Approved"].value_counts(normalize=True).min()
if class_Balance_check<0.30:
    print("imbalance class- need to be made balance\n")
else:
    print("its ok the class and balanced\n")

its ok the class and balanced



In [24]:
# 9. Feature Engineering (no leakage)
df["DebtToIncome"]=df["LoanAmount"]/df["Income"].replace(0, np.nan)
df["IncomePerYearEmployed"]=df["Income"]/(df["EmploymentYears"] + 1)

In [25]:
# 10. Feature Scaling (Research & Choose)
rbst_sclaer=RobustScaler()
no_scale={"Approved", "HasCollateral", "PreviousDefaults"}
numeric_cols=df.select_dtypes(include=["int64", "float64"]).columns.tolist()
to_scale=[col for col in numeric_cols if col not in no_scale]
df[to_scale]=rbst_sclaer.fit_transform(df[to_scale])

In [26]:
# 11. Final Checks & Save
print("After filling missing values of computing everything:\n")
print(df.head(5))
print(df.info())
print(df.isnull().sum())

OUTPUT_PATH=r"D:\Downloads\DS-ML COURSE\Jupiter(DS-ML)\clean_loan_dataset.csv"
df.to_csv(OUTPUT_PATH, index=False)
print("\nSaved!\nand Done the preprocessing!.")

After filling missing values of computing everything:

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  0.822149    -0.653722        -0.978632    0.246080              1   
1  0.564574    -0.737864         0.209402   -0.458384              1   
2 -0.856268     0.000000        -0.611111   -0.414958              0   
3 -0.911156    -0.498382         0.431624   -0.661037              1   
4 -0.649046    -0.718447        -0.235043   -0.482509              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0     -0.445244               8.274536  
1                 0         1     -0.912749               0.153994  
2                 0         1      0.280113              -0.126321  
3                 0         1     -0.315175              -0.669033  
4                 0         1     -0.284688              -0.284975  
<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 9 columns):


In [27]:
"""
Scaler Choice: RobustScaler

I chose `RobustScaler` instead of `StandardScaler` for this loan dataset
because two key numeric columns — `LoanAmount` (skewness ≈ 4.0) and
`Income` (skewness ≈ 1.4) — show significant right skew even after IQR
capping. `RobustScaler` centers using the median and scales using the
interquartile range (IQR) rather than mean and standard deviation, making
it less sensitive to the remaining distributional asymmetry in these
columns. This is more appropriate than `StandardScaler` when the
underlying distribution is not approximately normal.

`CreditScore` and `EmploymentYears` are near-normal (skewness ≈ 0.2 and
-0.03 respectively), so RobustScaler still handles them correctly — it
simply behaves similarly to StandardScaler for already-normal columns.

Source: scikit-learn documentation —
[RobustScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html)
"""

'\nScaler Choice: RobustScaler\n\nI chose `RobustScaler` instead of `StandardScaler` for this loan dataset\nbecause two key numeric columns — `LoanAmount` (skewness ≈ 4.0) and\n`Income` (skewness ≈ 1.4) — show significant right skew even after IQR\ncapping. `RobustScaler` centers using the median and scales using the\ninterquartile range (IQR) rather than mean and standard deviation, making\nit less sensitive to the remaining distributional asymmetry in these\ncolumns. This is more appropriate than `StandardScaler` when the\nunderlying distribution is not approximately normal.\n\n`CreditScore` and `EmploymentYears` are near-normal (skewness ≈ 0.2 and\n-0.03 respectively), so RobustScaler still handles them correctly — it\nsimply behaves similarly to StandardScaler for already-normal columns.\n\nSource: scikit-learn documentation —\n[RobustScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html)\n'